## Assignment 05 _ Population

**Author** - Alec Reid

**Date** - 11.11.2025

----

**Task**


**Part 1** 70%

Write a jupyter notebook that analyses the differences between the sexes by age in Ireland.
- Weighted mean age (by sex)
- The difference between the sexes by age

This part does not need to look at the regions. (ie You can take the notebook I used in the lectures and substitute the sexes for the regions.)

**Part 2** 20%
- In the same notebook, make a variable that stores an age (say 35).
- Write that code that would group the people within 5 years of that age together, into one age group 
- Calculate the population difference between the sexes in that age group.

**Part 3** 10%
- In the same notebook. Write the code that would work out which region in Ireland has the biggest population difference between the sexes in that age group

------

#### (1.) Background Info and Tidy Up

In [3]:
#Imports
import pandas as pd
import numpy as np

In [4]:
# Read Data - Show top 5 coulmns to get a sense of the data
url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/FY006A/CSV/1.0/en"
df = pd.read_csv(url)
df.head()

,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
0,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,IE0,Ireland,Number,5149139
1,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-1492-13a3-e055-000000000001,Carlow County Council,Number,61968
2,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-1433-13a3-e055-000000000001,Dublin City Council,Number,592713
3,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-149f-13a3-e055-000000000001,Dún Laoghaire Rathdown County Council,Number,233860
4,FY006AC01,Population,2022,2022,-,Both sexes,-,All ages,2ae19629-14a0-13a3-e055-000000000001,Fingal County Council,Number,330506


In [5]:
# Show the number of rows
len(df)

9792

In [6]:
# Find Unique values for sex
df['Sex'].unique()

array(['Both sexes', 'Male', 'Female'], dtype=object)

In [7]:
# Find unique values for age
df['Single Year of Age'].unique()

array(['All ages', 'Under 1 year', '1 year', '2 years', '3 years',
       '4 years', '5 years', '6 years', '7 years', '8 years', '9 years',
       '10 years', '11 years', '12 years', '13 years', '14 years',
       '15 years', '16 years', '17 years', '18 years', '19 years',
       '20 years', '21 years', '22 years', '23 years', '24 years',
       '25 years', '26 years', '27 years', '28 years', '29 years',
       '30 years', '31 years', '32 years', '33 years', '34 years',
       '35 years', '36 years', '37 years', '38 years', '39 years',
       '40 years', '41 years', '42 years', '43 years', '44 years',
       '45 years', '46 years', '47 years', '48 years', '49 years',
       '50 years', '51 years', '52 years', '53 years', '54 years',
       '55 years', '56 years', '57 years', '58 years', '59 years',
       '60 years', '61 years', '62 years', '63 years', '64 years',
       '65 years', '66 years', '67 years', '68 years', '69 years',
       '70 years', '71 years', '72 years', '73 years', '

--------

#### (2.) Mean Age of Each Sex

In [8]:
# Load data directly from CSO API
url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/FY006A/CSV/1.0/en"
df = pd.read_csv(url)

In [9]:

# Keep only required columns
df = df[['Sex', 'Single Year of Age', 'VALUE']]

# Remove "All ages"
df = df[df['Single Year of Age'] != 'All ages']

# Clean age column
df['Single Year of Age'] = (
    df['Single Year of Age']
    .replace('Under 1 year', '0')
    .str.replace(r'\D', '', regex=True)
    .astype(int)
)

# Ensure VALUE is numeric
df['VALUE'] = pd.to_numeric(df['VALUE'], errors='coerce')

# Calculate weighted mean age by sex
mean_age_by_sex = (
    df.groupby('Sex')
      .apply(lambda x: np.average(x['Single Year of Age'], weights=x['VALUE']))
      .reset_index(name='Mean Age')
)

# Round for presentation
mean_age_by_sex['Mean Age'] = mean_age_by_sex['Mean Age'].round(1)

print(mean_age_by_sex)

          Sex  Mean Age
0  Both sexes      38.3
1      Female      38.9
2        Male      37.7


#### (3.) Differences between the sexes by Age

I'm going to take this to mean the number of males or females per single year of age - makes sense to me to put them into their own pivot table

In [ ]:
# Keep relevant columns
df = df[['Sex', 'Single Year of Age', 'VALUE']]

# Drop the 'Both sexes' category
df = df[df['Sex'] != 'Both sexes']

# Remove summary rows
df = df[df['Single Year of Age'] != 'All ages']

# Ensure VALUE is numeric
df['VALUE'] = pd.to_numeric(df['VALUE'], errors='coerce')

# Create table: rows = age, columns = sex
age_sex_table = (
    df.pivot_table(
        index='Single Year of Age',
        columns='Sex',
        values='VALUE',
        aggfunc='sum'
    )
    .sort_index()
)

print(age_sex_table)

# show last 40 rows (60yrs-100yrs)
age_sex_table.tail(40)  


Sex                 Female   Male
Single Year of Age               
0                    56372  59220
1                    55090  57750
2                    57948  60472
3                    58966  62002
4                    59638  63372
...                    ...    ...
96                    1912    654
97                    1464    434
98                     984    260
99                     672    210
100                   1168    308

[101 rows x 2 columns]


Sex,Female,Male
Single Year of Age,,
61,57712,55698
62,55798,54232
63,52624,51768
64,52218,50254
65,50366,49800
66,49324,47518
67,49048,47254
68,46036,45004
69,46750,45188


In [18]:
# Filter rows where the number of males exceeds the number of females
males_exceed_females = age_sex_table[age_sex_table['Male'] > age_sex_table['Female']]

print(males_exceed_females)

Sex                 Female   Male
Single Year of Age               
0                    56372  59220
1                    55090  57750
2                    57948  60472
3                    58966  62002
4                    59638  63372
5                    62684  65368
6                    65244  68184
7                    67144  70858
8                    68874  72592
9                    70454  73938
10                   72132  76070
11                   73726  76592
12                   74594  77914
13                   73564  76948
14                   72160  74704
15                   68984  71896
16                   66336  69292
17                   66090  69962
18                   65704  67800
19                   63458  65734
20                   63248  64240
21                   59780  61542
22                   61536  62900
23                   60102  61742
24                   58728  60468
25                   58588  59938
26                   57728  58752
53            

Thats nuts, after the age of 53 there is always more women per single year of age class. It's even more nuts as we start of with more men - for the first 22 years in fact!

#### (4.) Grouping Together Based on Variable Age 35

In [22]:
# Choose variable age
Var_age = 35

# Group ages based on 5 years over and under Var_age
age_sex_table['Age_Group'] = np.where(  
    age_sex_table.index < Var_age,
    f'Under {Var_age}',
    f'{Var_age} and over'
)   

# Aggregate population by the new age groups
grouped_population = (
    age_sex_table
    .groupby('Age_Group')

# ssum the number of males and female for target
    .sum()[['Male', 'Female']]
)
print(grouped_population)


Sex             Male   Female
Age_Group                    
35 and over  2781708  2946392
Under 35     2307390  2262788


Kinda snookered myself here after I created the 'age_sex' table, now not sure how to replate this back to original dataframe to work out location 